# Reducing Model Size with Quantization

A trained model lives in float32 — 4 bytes per parameter. A 7B model occupies ~28 GB. Quantization reduces numerical precision to shrink the model and speed up inference, with surprisingly little quality loss if done carefully. The key insight is that for language model weights, [relative ordering matters far more than absolute precision]{.mark}: a weight vector scaled by 0.01 and quantized to 256 levels retains the ranking structure that determines which features fire, even though the exact floating-point values are lost. This notebook covers the arithmetic of quantization, post-training quantization (PTQ), activation calibration, INT4 grouped quantization, and quantization-aware training (QAT).

## The Numerical Argument

A float32 number has 1 sign bit, 8 exponent bits, and 23 mantissa bits — about 7 decimal digits of precision. For LLM weights, the exponent bits determine magnitude and the mantissa bits determine fine-grained precision.

For inference, consider what a weight actually does: it scales and shifts activations. The *relative* ordering of weights in a row matters more than their precise values; the model's output is dominated by the top-magnitude weights, and small-magnitude weights contribute negligibly regardless of whether they are represented with 1 bit or 32 bits of mantissa.

This motivates the core observation: **weight quantization** is nearly lossless at 8 bits, acceptable at 4 bits, and starts hurting quality at 3 bits and below. **Activation quantization** is much harder because activation tensors contain outliers — rare dimensions with values 100× the median — that force the quantization scale to be large and waste precision on common values.

## Quantization Arithmetic

The affine (asymmetric) quantization map from float to integer and back:

$$x_q = \text{clip}\!\left(\text{round}\!\left(\frac{x}{s}\right) + z,\; -2^{b-1},\; 2^{b-1}-1\right), \qquad \hat{x} = s \cdot (x_q - z),$$

where $s > 0$ is the **scale**, $z$ is the **zero-point** (an integer offset), and $b$ is the bit width. The maximum representable error is $s/2$ — one half of a quantization step.

**Symmetric quantization** sets $z = 0$: the integer range $[-n_\text{levels}, n_\text{levels}]$ is centered at zero. This is preferred for LLM weights because weight distributions are approximately Gaussian and centered near zero. With $z = 0$:

$$s = \frac{\max |x|}{n_\text{levels}}, \qquad n_\text{levels} = 2^{b-1} - 1.$$

Implementing symmetric per-tensor quantization and measuring error:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass


def quantize_symmetric(
    x: torch.Tensor,
    n_bits: int = 8,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Symmetric per-tensor quantization.

    Args:
        x: float tensor to quantize.
        n_bits: bit width (e.g. 8 for INT8).

    Returns:
        x_q: quantized integer tensor (int8 for n_bits <= 8).
        scale: float scale factor.
    """
    n_levels = 2 ** (n_bits - 1) - 1   # 127 for INT8
    abs_max = x.abs().max().clamp(min=1e-8)
    scale = abs_max / n_levels
    x_q = torch.clamp(
        torch.round(x / scale),
        -n_levels, n_levels
    ).to(torch.int8 if n_bits <= 8 else torch.int16)
    return x_q, scale


def dequantize_symmetric(
    x_q: torch.Tensor,
    scale: torch.Tensor,
) -> torch.Tensor:
    """Recover float approximation from quantized tensor."""
    return x_q.float() * scale


def quantization_error(x: torch.Tensor, n_bits: int = 8) -> dict:
    """Measure quantization error statistics at a given bit width.

    Args:
        x: float tensor.
        n_bits: bit width.

    Returns:
        Dict with 'max_error', 'mean_error', 'rmse', 'snr_db', 'scale'.
    """
    x_q, scale = quantize_symmetric(x, n_bits)
    x_hat = dequantize_symmetric(x_q, scale)
    err = (x - x_hat).abs()
    snr = x.pow(2).mean() / (err.pow(2).mean() + 1e-10)
    return {
        "max_error":  err.max().item(),
        "mean_error": err.mean().item(),
        "rmse":       err.pow(2).mean().sqrt().item(),
        "snr_db":     10 * torch.log10(snr).item(),
        "scale":      scale.item(),
    }


# Typical LLM weight magnitude
torch.manual_seed(42)
w = torch.randn(768, 768) * 0.02

for bits in [8, 6, 4, 3, 2]:
    stats = quantization_error(w, bits)
    print(
        f"INT{bits}: rmse={stats['rmse']:.6f}  "
        f"snr={stats['snr_db']:.1f} dB  "
        f"scale={stats['scale']:.6f}"
    )

### Per-tensor vs per-channel

**Per-tensor quantization** uses a single scale for all weights in a layer. If different output channels have very different magnitudes — common in practice — the large-magnitude channels force a coarse scale, wasting precision on small-magnitude channels.

[**Per-channel quantization** uses a separate scale per output channel, set by each channel's own max absolute value.]{.mark} More accurate, at the cost of storing $d_{\text{out}}$ scale factors per layer instead of one.

Implementing per-channel quantization:

In [ ]:
def quantize_per_channel(
    weight: torch.Tensor,
    n_bits: int = 8,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Per-output-channel symmetric quantization.

    Args:
        weight: shape (d_out, d_in).
        n_bits: bit width.

    Returns:
        w_q: (d_out, d_in) quantized weights (int8).
        scales: (d_out,) per-channel scale factors.
    """
    n_levels = 2 ** (n_bits - 1) - 1
    abs_max = weight.abs().max(dim=1).values.clamp(min=1e-8)  # (d_out,)
    scales = abs_max / n_levels                               # (d_out,)
    w_scaled = weight / scales.unsqueeze(1)                   # (d_out, d_in)
    w_q = torch.clamp(torch.round(w_scaled), -n_levels, n_levels).to(torch.int8)
    return w_q, scales


def dequantize_per_channel(
    w_q: torch.Tensor,
    scales: torch.Tensor,
) -> torch.Tensor:
    """Dequantize per-channel INT8 weights.

    Args:
        w_q: shape (d_out, d_in) int8.
        scales: shape (d_out,).

    Returns:
        Float weight tensor of shape (d_out, d_in).
    """
    return w_q.float() * scales.unsqueeze(1)

## Post-Training Quantization (PTQ)

PTQ quantizes a trained model without any additional training. The simplest scheme: replace every `nn.Linear` with a `QuantizedLinear` that stores weights as INT8 and dequantizes to float before the matmul. This gives 4× memory reduction (1 byte vs 4 bytes per parameter) with negligible quality loss at 8 bits.

Implementing `QuantizedLinear` and a drop-in model quantization function:

In [ ]:
class QuantizedLinear(nn.Module):
    """INT8 weight-quantized linear layer.

    Weights stored as int8; dequantized to float at forward time.
    Memory: 1 byte/param vs 4 bytes for float32 = 4× compression.

    Args:
        in_features: input dimension.
        out_features: output dimension.
        bias: whether to include a bias term.
        n_bits: quantization bit width.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = True,
        n_bits: int = 8,
    ) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.n_bits = n_bits
        self.register_buffer("weight_q", torch.zeros(out_features, in_features, dtype=torch.int8))
        self.register_buffer("scales", torch.ones(out_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

    @classmethod
    def from_linear(cls, linear: nn.Linear, n_bits: int = 8) -> "QuantizedLinear":
        """Quantize an existing nn.Linear layer in-place.

        Args:
            linear: the source nn.Linear module.
            n_bits: target bit width.

        Returns:
            A new QuantizedLinear with quantized weights.
        """
        layer = cls(
            linear.in_features, linear.out_features,
            bias=(linear.bias is not None), n_bits=n_bits,
        )
        w_q, scales = quantize_per_channel(linear.weight.data, n_bits)
        layer.weight_q.copy_(w_q)
        layer.scales.copy_(scales)
        if linear.bias is not None:
            layer.bias.data.copy_(linear.bias.data)
        return layer

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weight_fp = dequantize_per_channel(self.weight_q, self.scales)  # <1>
        return F.linear(x, weight_fp.to(x.dtype), self.bias)


def quantize_model_ptq(
    model: nn.Module,
    n_bits: int = 8,
    skip_modules: tuple = ("lm_head",),
) -> nn.Module:
    """Replace all nn.Linear layers with QuantizedLinear.

    Args:
        model: the model to quantize (modified in-place).
        n_bits: bit width for quantization.
        skip_modules: names of modules to skip (e.g. output projection).

    Returns:
        The quantized model.
    """
    for name, module in list(model.named_modules()):
        if any(skip in name for skip in skip_modules):  # <2>
            continue
        if isinstance(module, nn.Linear):
            parent_name, child_name = name.rsplit(".", 1) if "." in name else ("", name)
            parent = model if not parent_name else dict(model.named_modules())[parent_name]
            setattr(parent, child_name, QuantizedLinear.from_linear(module, n_bits))
    return model

1. Dequantize at forward time: weights are stored as int8, converted to float for the matmul. Memory usage is reduced; compute is standard float.
2. The `lm_head` (output projection to vocabulary) is skipped — quantizing it disproportionately hurts perplexity because it maps to a 50K+ vocabulary with many near-zero logit differences.

## Calibration

Calibration runs a small set of representative inputs through the model and collects statistics about activation ranges. For weight-only quantization (PTQ above), calibration is not strictly needed — weight ranges are fixed after training. For activation quantization, calibration determines the scale factors for intermediate tensors.

Calibration is also the foundation of more advanced methods like GPTQ and SmoothQuant, which adjust weight quantization based on activation statistics to minimize end-to-end quantization error.

Implementing `ActivationCalibrator` using forward hooks:

In [ ]:
class ActivationCalibrator:
    """Collect min/max activation statistics over a calibration dataset.

    Hooks into all Linear layers and records input/output ranges.
    Used to set activation quantization scales for full INT8 quantization.

    Args:
        model: the model to calibrate.
    """

    def __init__(self, model: nn.Module) -> None:
        self.stats: dict = {}
        self.hooks: list = []
        self._register_hooks(model)

    def _register_hooks(self, model: nn.Module) -> None:
        for name, module in model.named_modules():
            if isinstance(module, (nn.Linear, QuantizedLinear)):
                def make_hook(n):
                    def hook(mod, inp, out):
                        x = inp[0].detach().float()
                        o = out.detach().float()
                        if n not in self.stats:
                            self.stats[n] = {
                                "input_min":  x.min().item(),
                                "input_max":  x.max().item(),
                                "output_min": o.min().item(),
                                "output_max": o.max().item(),
                                "n_batches":  1,
                            }
                        else:
                            s = self.stats[n]
                            s["input_min"]  = min(s["input_min"],  x.min().item())
                            s["input_max"]  = max(s["input_max"],  x.max().item())
                            s["output_min"] = min(s["output_min"], o.min().item())
                            s["output_max"] = max(s["output_max"], o.max().item())
                            s["n_batches"] += 1
                    return hook
                self.hooks.append(module.register_forward_hook(make_hook(name)))

    def calibrate(
        self,
        model: nn.Module,
        dataloader,
        device: torch.device,
        n_batches: int = 50,
    ) -> dict:
        """Run calibration forward passes and collect statistics.

        Args:
            model: model to calibrate.
            dataloader: yields (input_ids, labels) batches.
            device: compute device.
            n_batches: number of batches to run.

        Returns:
            stats: dict mapping layer name to activation range statistics.
        """
        model.eval()
        with torch.no_grad():
            for i, (x, _) in enumerate(dataloader):
                if i >= n_batches:
                    break
                model(x.to(device))
        self.remove_hooks()
        return self.stats

    def remove_hooks(self) -> None:
        """Remove all registered forward hooks."""
        for h in self.hooks:
            h.remove()
        self.hooks.clear()

## INT4 Weight Quantization

4-bit quantization halves memory again vs INT8. With 4 bits, each weight can take only 16 distinct values — intuitively this seems too coarse. But in practice, for large models, the effect on perplexity is modest because individual weights have small effect on output (errors average across many weights) and because **block (group) quantization** uses a separate scale per small group of weights.

### Block quantization

Instead of one scale per output channel, use one scale per **group** of $g$ consecutive input-dimension weights:

```
Channel 0: [w_0, ..., w_{g-1}] → scale_0
           [w_g, ..., w_{2g-1}] → scale_1
           ...
```

Common group sizes: $g = 32, 64, 128$. Smaller groups give more accurate quantization at the cost of more scale factors to store. $g = 128$ is the standard for GPTQ-style INT4.

In [ ]:
def quantize_int4_grouped(
    weight: torch.Tensor,
    group_size: int = 128,
) -> tuple[torch.Tensor, torch.Tensor]:
    """INT4 grouped quantization.

    Each group of `group_size` weights along the input dimension shares
    one scale factor. Weights quantized to 4-bit signed range [-8, 7],
    stored as int8.

    Args:
        weight: shape (d_out, d_in).
        group_size: number of weights per group.

    Returns:
        w_q: (d_out, d_in) int8 — quantized weights.
        scales: (d_out, d_in // group_size) per-group scale factors.
    """
    d_out, d_in = weight.shape
    assert d_in % group_size == 0, (
        f"d_in={d_in} must be divisible by group_size={group_size}"
    )
    n_groups = d_in // group_size
    n_levels = 7   # INT4 signed: range [-8, 7]

    w_grouped = weight.view(d_out, n_groups, group_size)           # <1>
    abs_max = w_grouped.abs().max(dim=2).values.clamp(min=1e-8)   # (d_out, n_groups)
    scales = abs_max / n_levels

    w_scaled = w_grouped / scales.unsqueeze(2)                    # (d_out, n_groups, group_size)
    w_q = torch.clamp(torch.round(w_scaled), -8, 7).to(torch.int8)
    return w_q.view(d_out, d_in), scales


def dequantize_int4_grouped(
    w_q: torch.Tensor,
    scales: torch.Tensor,
    group_size: int = 128,
) -> torch.Tensor:
    """Dequantize INT4 grouped weights.

    Args:
        w_q: shape (d_out, d_in) int8.
        scales: shape (d_out, n_groups).
        group_size: number of weights per group.

    Returns:
        Float weight tensor of shape (d_out, d_in).
    """
    d_out, d_in = w_q.shape
    n_groups = d_in // group_size
    w_grouped = w_q.view(d_out, n_groups, group_size).float()
    w_fp = w_grouped * scales.unsqueeze(2)
    return w_fp.view(d_out, d_in)

Annotation:

1. Reshape to expose the group structure: each group of `group_size` contiguous input-dimension weights becomes a separate slice along axis 2.

### Why activations stay in float

The key obstacle to activation quantization is **outliers**. Research on LLM activations (LLM.int8() [@dettmers2022]) found that a small fraction (~0.1%) of activation dimensions have values 100× larger than the median. These outliers force the quantization scale to be large, wasting the entire 4-bit range on rare extreme values while common values all map to the same few integers.

[Weights do not have this problem: weight distributions are approximately Gaussian with few outliers]{.mark} — weight decay regularizes against large weights. This is the fundamental reason why weight-only quantization works far better than full quantization at the same bit width.

## Quantization-Aware Training (QAT)

PTQ quantizes after training — the weights were never optimized to be robust to quantization noise. QAT inserts quantization into the training loop: the forward pass simulates quantization, and the backward pass trains weights that are inherently more quantizable.

### Fake quantization and the straight-through estimator

During the forward pass we apply the full quantize-dequantize cycle, producing $\hat{x} \approx x$ with quantization noise. During the backward pass we face a problem: `round()` has zero gradient almost everywhere. The [straight-through estimator (STE)]{.underline} simply passes the gradient through the rounding operation as if it were the identity:

$$\frac{\partial \hat{x}}{\partial x} \approx \mathbf{1}\!\left[\left|\frac{x}{s}\right| \leq n_{\text{levels}}\right].$$

Within the representable range the gradient passes through unchanged; outside the clamp range the gradient is zero (honest clamp gradient).

In [ ]:
class FakeQuantize(torch.autograd.Function):
    """Forward: quantize then dequantize (introduces quantization noise).
    Backward: straight-through estimator.
    """

    @staticmethod
    def forward(ctx, x, scale, n_bits):                        # <1>
        n_levels = 2 ** (n_bits - 1) - 1
        x_scaled = x / scale
        ctx.save_for_backward((x_scaled.abs() <= n_levels).float())
        x_q = torch.clamp(torch.round(x_scaled), -n_levels, n_levels)
        return x_q * scale

    @staticmethod
    def backward(ctx, grad_output):                            # <2>
        mask, = ctx.saved_tensors
        return grad_output * mask, None, None


def fake_quantize(
    x: torch.Tensor, scale: torch.Tensor, n_bits: int
) -> torch.Tensor:
    """Apply fake quantization (quantize+dequantize with STE backward)."""
    return FakeQuantize.apply(x, scale, n_bits)


class QATLinear(nn.Module):
    """Linear layer with fake quantization in the forward pass.

    Weights are trained in float but experience quantization noise
    during each forward pass, making them robust to PTQ at inference.

    Args:
        in_features: input dimension.
        out_features: output dimension.
        bias: whether to include a bias term.
        n_bits: target quantization bit width.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = True,
        n_bits: int = 8,
    ) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.n_bits = n_bits
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.kaiming_uniform_(self.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training:
            n_levels = 2 ** (self.n_bits - 1) - 1
            abs_max = self.weight.abs().max(dim=1).values.clamp(min=1e-8)
            scales = abs_max / n_levels                        # <3>
            scale_col = scales.unsqueeze(1).expand_as(self.weight)
            weight_fq = fake_quantize(self.weight, scale_col, self.n_bits)
        else:
            weight_fq = self.weight                            # <4>
        return F.linear(x, weight_fq, self.bias)

1. Forward: quantize to integer grid and dequantize back to float, introducing quantization noise. Saves the in-range mask for backward.
2. Backward: STE passes the gradient unchanged through `round()`; zeros the gradient for out-of-range values (where the clamp correctly has zero gradient).
3. Per-channel scale computed from current weight values — adapts dynamically during training.
4. At inference, use the real float weights and then apply PTQ once training is done.

### When is QAT worth the cost?

QAT requires re-running fine-tuning with quantization noise — typically 10–20% of the original training compute. The quality improvement over PTQ is significant at 4-bit but modest at 8-bit:

| Method | INT8 perplexity delta | INT4 perplexity delta |
|---|---|---|
| PTQ (no calibration) | +0.5–2% | +5–15% |
| PTQ (calibrated) | +0.1–0.5% | +2–8% |
| QAT | +0.05–0.2% | +0.5–2% |

: Quantization quality vs method. {tbl-colwidths="[40, 30, 30]"}

[For INT8: PTQ with calibration is almost always sufficient. For INT4: QAT is worth considering for production models.]{.mark}

## Measuring Quality Degradation

Perplexity is the standard metric for quantization quality. We measure it on a held-out corpus at each bit width to produce a bit-width vs perplexity tradeoff curve.

Two additional diagnostics: model size in MB and the fraction of parameters that are quantized.

In [ ]:
@torch.no_grad()
def evaluate_perplexity(
    model: nn.Module,
    dataloader,
    device: torch.device,
    n_batches: int = 100,
) -> float:
    """Compute perplexity on a dataloader.

    Args:
        model: language model; forward pass takes (input_ids, labels) and
               returns (logits, loss).
        dataloader: yields (input_ids, labels) batches.
        device: compute device.
        n_batches: how many batches to evaluate.

    Returns:
        Perplexity = exp(mean cross-entropy loss).
    """
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for i, (x, y) in enumerate(dataloader):
        if i >= n_batches:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        n_tokens = (y != -100).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens

    avg_loss = total_loss / max(total_tokens, 1)
    return float(np.exp(avg_loss))


def model_size_mb(model: nn.Module) -> float:
    """Total size of all parameters and buffers in MB."""
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_bytes + buffer_bytes) / 1e6


def quantization_summary(model: nn.Module) -> None:
    """Print a per-layer quantization summary."""
    print(f"\nQuantization Summary")
    print("─" * 60)
    total_params = 0
    quant_params = 0

    for name, module in model.named_modules():
        if isinstance(module, (nn.Linear, QuantizedLinear)):
            n = (
                module.weight_q if hasattr(module, "weight_q") else module.weight
            ).numel()
            is_q = isinstance(module, QuantizedLinear)
            bits = module.n_bits if hasattr(module, "n_bits") else 32
            total_params += n
            if is_q:
                quant_params += n
            flag = "Q" if is_q else "F"
            print(f"  {flag} {name:45s}  {n/1e3:6.1f}K  INT{bits if is_q else 32}")

    print("─" * 60)
    print(f"  Total params:      {total_params/1e6:.2f}M")
    print(
        f"  Quantized params:  {quant_params/1e6:.2f}M  "
        f"({100 * quant_params / max(total_params, 1):.1f}%)"
    )
    print(f"  Model size:        {model_size_mb(model):.1f} MB")

## Summary

| Concept | Key detail |
|---|---|
| Why weights tolerate low precision | Relative ordering matters more than absolute value; errors average across many weights |
| Affine mapping | $x_q = \text{round}(x/s) + z$, $\hat{x} = s(x_q - z)$; error bounded by $s/2$ |
| Symmetric quantization | $z=0$, signed range; preferred for LLM weights (near-symmetric distribution) |
| Per-channel scaling | One scale per output channel; adapts to channel-wise magnitude variation |
| Per-group (block) scaling | One scale per $g$ weights within a channel; standard for INT4 ($g=128$) |
| PTQ | Quantize after training; no training needed; INT8 near-lossless |
| Calibration | Needed for activation quantization; collect min/max over representative inputs |
| INT4 grouped | 4-bit range $[-8, 7]$; group size 128; 4× memory vs FP32 |
| Why activations stay float | Outlier dimensions force coarse grid; weight-only avoids this problem |
| Straight-through estimator | $\partial\hat{x}/\partial x \approx 1$ through round; zero outside clamp range |
| QAT vs PTQ | QAT worth it for INT4; PTQ + calibration is sufficient for INT8 |
| Skip `lm_head` | Output projection quantization disproportionately hurts perplexity |

: Quantization reference. {tbl-colwidths="[35, 65]"}

## Exercises

1. **Bit-width sweep.** Run `quantization_error` for `torch.randn(768, 768) * 0.02` at bits $\in \{2, 3, 4, 6, 8\}$. Plot RMSE vs bit width on a log-log scale. Where is the "knee" — the bit width where error starts growing rapidly? Does it shift if you add an outlier: `w[0, 0] = 10.0`?

2. **Per-tensor vs per-channel.** Quantize the same weight matrix with per-tensor and per-channel INT8. Compute the RMSE of each and plot the per-channel weight magnitude distribution. Confirm that per-channel is always at least as accurate, with the gap growing when channel magnitudes vary.

3. **`QuantizedLinear` memory.** Instantiate a `QuantizedLinear(1024, 1024)` and compare `model_size_mb` against the equivalent `nn.Linear`. Verify the 4× compression. What is the overhead from storing the scale factors?

4. **Group size ablation.** Apply `quantize_int4_grouped` with group sizes $\in \{32, 64, 128, 256\}$ to a weight matrix of shape `(4096, 4096)`. Plot RMSE vs group size and total storage (weights + scales) vs group size. What is the Pareto-optimal group size?

5. **STE gradient verification.** Implement a toy 1D regression where a single weight is fake-quantized. Check that gradients flow correctly through `FakeQuantize` using `torch.autograd.gradcheck`. Confirm the gradient is zero for weights outside the representable range.

6. **QAT vs PTQ comparison.** Train a small MLP on a regression task (a) normally and then apply PTQ, (b) with `QATLinear` replacing all linear layers. Compare final test loss. Does QAT help more at INT8 or INT4?

■